# OSAI Project 1 — Colab v1 Baseline Reproduction

**목적**: Pascal-VOC 21-class semantic segmentation v1 baseline (ResNet-50 + DeepLabV3+, OS=16) 재현. PDF 정책 "Colab T4/L4 처음부터 전체 파이프라인 재실행 = 제출물" 준수.

**소요 시간 (T4 기준)**:
- 셋업 + 데이터 다운로드: ~30–45분
- COCO mask cache: ~10–15분
- 학습 (Stage 1 80K + Stage 2 8K): **~10–12시간**
- 평가 + 추론 + 제출 패키징: ~40–50분
- **총 ~12–14시간** (Colab Pro+ 단일 세션 안에 fit)

**산출물 (Drive에 저장)**:
- `model_structure.onnx` → 채점 사이트 ONNX 업로드
- `submission_pred.zip` → 채점 사이트 PNG 업로드
- `checkpoints/model.pth` → 학교 사이트 코드베이스 zip 포함

## 사전 준비 (1회)

1. **Colab Pro+ 활성화** (단일 세션 24h 한도 필요)
2. **GPU 활성화**: 메뉴 → 런타임 → 런타임 유형 변경 → **T4** 또는 **L4** 선택
3. **Google Drive 구조 생성**:
   ```
   /MyDrive/osai-p1/
   └── img.zip          ← 데스크탑에서 1회 업로드 (1000 test images, 114MB)
   ```
4. **WandB API key 준비**: https://wandb.ai/authorize
5. 이 노트북을 Colab으로 열고 셀 순서대로 실행 (또는 "Run All")

## 1. Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 설정 변수

필요시 수정. 기본값은 v1 baseline + Drive 표준 위치.

In [ ]:
REPO_URL = "https://github.com/geniemo/osai.git"
BRANCH   = "main"                       # v1 baseline = main 브랜치
CONFIG   = "src/config/colab.yaml"      # Drive 경로 + Stage 1 80K + Stage 2 8K
DRIVE    = "/content/drive/MyDrive/osai-p1"

import os
os.makedirs(f"{DRIVE}/checkpoints", exist_ok=True)
print(f"Repo: {REPO_URL} (branch={BRANCH})")
print(f"Config: {CONFIG}")
print(f"Drive: {DRIVE}")
assert os.path.exists(f"{DRIVE}/img.zip"), f"❌ img.zip not found at {DRIVE}/img.zip — Drive에 업로드 후 다시 실행"

## 3. 저장소 clone

In [ ]:
%cd /content
!rm -rf osai
!git clone --branch {BRANCH} --depth 1 {REPO_URL} osai
%cd osai/p1

## 4. uv 설치 + 의존성 sync

PyTorch 2.7+ + CUDA 12.8 binary 자동 설치. ~3–5분.

In [ ]:
!pip install -q uv
!uv sync

## 5. GPU 확인

**기대**: Tesla T4 (capability 7,5) 또는 NVIDIA L4 (8,9). 다른 GPU면 spec 위반.

In [ ]:
!uv run python -c "\
import torch; \
print('Device:', torch.cuda.get_device_name(0)); \
print('Capability:', torch.cuda.get_device_capability(0)); \
print('Memory:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB'); \
print('PyTorch:', torch.__version__)"

## 6. WandB 로그인

API key 입력 (https://wandb.ai/authorize 에서 복사). Colab Secrets에 저장돼 있으면 자동 사용.

In [ ]:
# Option A: 환경변수로 (Colab Secrets 사용 권장)
# from google.colab import userdata
# import os; os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')

# Option B: interactive prompt (Run 후 key 붙여넣기)
!uv run wandb login

## 7. Test images 복사 (Drive → Colab)

In [ ]:
!mkdir -p input/test_public
!cp {DRIVE}/img.zip input/
!cd input && unzip -q -o img.zip -d test_public && rm img.zip
# img.zip은 flat 구조 (test_public/ 폴더 없음) → test_public/ 안에 풀기
!ls input/test_public | wc -l   # 1000 expected
!ls input/test_public | head -3

## 8. 데이터 다운로드 (VOC + COCO)

VOC ~2GB (5분), COCO ~25GB (20–40분). 압축 해제 자동.

In [ ]:
!uv run python -m src.data.download --voc-root data/voc --coco-root data/coco

## 9. COCO mask cache 사전 생성

95K instance mask를 PNG로 1회 생성 (~10–15분). 학습 시 cache hit으로 빠름.

In [ ]:
!uv run python -m src.build_coco_masks --coco-root data/coco --num-workers 4

## 10. 학습 (Stage 1 + Stage 2 자동) — ~10–12시간

ckpt가 Drive에 저장됨 (`{DRIVE}/checkpoints/`). 세션 끊기면 새 세션에서 셀 1–9 다시 + 이 셀 다시 → resume.

WandB 실시간 모니터링: https://wandb.ai/g1nie-sungkyunkwan-university/osai-p1-local

In [ ]:
!uv run python -m src.train --config {CONFIG}

## 11. Validation mIoU 측정

최종 ckpt로 val mIoU + 21 class별 IoU 출력.

In [ ]:
!uv run python -m src.eval --config {CONFIG} --ckpt {DRIVE}/checkpoints/model.pth

## 12. TTA Validation mIoU (선택, 실제 inference 성능)

Multi-scale + hflip TTA로 측정. ~3–5분 (T4).

In [ ]:
!uv run python -m src.eval_tta --config {CONFIG} --ckpt {DRIVE}/checkpoints/model.pth

## 13. 추론 — test_public 1000장 (TTA)

input/test_public/*.jpg → output/pred_FINAL/*.png. ~5–10분 (T4).

In [ ]:
!mkdir -p output
!uv run python -m src.infer \
    --config {CONFIG} \
    --ckpt {DRIVE}/checkpoints/model.pth \
    --input input/test_public \
    --output output/pred_FINAL

## 14. ONNX export (가중치 제거, 채점용 ONNX)

`model_structure.onnx` 생성 (~0.3MB, ≤10MB 채점 한도).

In [ ]:
!uv run python -m src.export_onnx \
    --config {CONFIG} \
    --ckpt {DRIVE}/checkpoints/model.pth \
    --out {DRIVE}/model_structure.onnx

## 15. FLOPs 측정 (채점 기준 확인)

In [ ]:
!uv run python -m src.measure_flops --onnx {DRIVE}/model_structure.onnx

## 16. PNG zip 패키징 (채점용 PNG 제출)

1000 PNG (000.png~999.png), 픽셀 [0,20], 압축해제 ≤500MB 검증 + zip 생성.

In [ ]:
!uv run python -m src.package_submission \
    --pred output/pred_FINAL \
    --out {DRIVE}/submission_pred.zip

## 17. 결과물 확인

Drive에 다음 3개 산출물 생성됨:
- `model_structure.onnx` (~0.3MB) — 채점 사이트 ONNX 업로드
- `submission_pred.zip` (~10–30MB) — 채점 사이트 PNG 업로드
- `checkpoints/model.pth` (~180MB) — 학교 사이트 코드베이스 zip 포함

WandB Overview에서 GPU evidence (`Tesla T4` 또는 `NVIDIA L4`) 자동 기록됨.

In [ ]:
!ls -la {DRIVE}/
!ls -la {DRIVE}/checkpoints/
!echo '\nWandB Overview에 GPU evidence 캡처해서 리포트에 첨부 필요'

## 채점 사이트 업로드

1. **PNG zip**: `{DRIVE}/submission_pred.zip` 다운로드 → 채점 사이트 PNG 슬롯
2. **ONNX**: `{DRIVE}/model_structure.onnx` 다운로드 → 채점 사이트 ONNX 슬롯

## 학교 사이트 zip (별도 작성)

데스크탑/로컬에서:
```bash
# Drive의 model.pth를 다운로드 후
cp ~/Downloads/model.pth p1/checkpoints/model.pth
# 리포트 PDF 준비
# zip 생성
zip -r p1/<학번>_project01.zip \
    p1/src p1/checkpoints/model.pth p1/submit \
    p1/<학번>_project01_report.pdf p1/pyproject.toml p1/README.md \
    -x '**/__pycache__/*' '**/.venv/*'
```

## 트러블슈팅

| 문제 | 해결 |
|---|---|
| Drive mount 실패 | 새 세션에서 다시. 인증 만료 가능 |
| `uv sync` CUDA wheel 오류 | 런타임 재시작 후 재시도 |
| OOM | colab.yaml의 `batch_size: 16` → `12`로 낮춤 |
| 세션 timeout (24h 후) | 새 세션에서 셀 1–9 다시 + 셀 10 → 자동 resume |
| WandB 로그인 실패 | Cell 6 다시 또는 환경변수 사용 |